In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.6 Molecules: Rotation, Vibration, and the Heat-Capacity Staircase

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.6",
    title="Molecules: Rotation, Vibration, and the Heat-Capacity Staircase",
    blurb="A diatomic gas is a bank of quantum switches. Rotation wakes near one "
    "temperature, vibration near another far above it, and the heat capacity climbs "
    "a staircase that equipartition, which cannot switch anything off, declared "
    "impossible. Hydrogen adds the deepest twist: the Pauli principle, acting on two "
    "protons, deletes half of each spin species' rotational levels, splits the gas "
    "into ortho and para varieties that barely interconvert, and thereby explains a "
    "famous experimental puzzle — and why rocket fuel must be catalyzed.",
    difficulty="advanced",
    estimate="190–230 min",
)

## Notebook overview

Movement I closes with molecules, and with the payoff its two previous notebooks were built to
fund. The qubit gave us one quantum switch (a gap that freezes out); the oscillator gave another,
plus the scale $\theta = \hbar\omega/k_B$ that decides when each switch flips. A diatomic molecule
wires several such switches in series: translation (classical at any laboratory temperature —
Volume V's $\tfrac32$, invoked), **rotation** with its own quantum $\theta_{\text{rot}} =
\hbar^2/2Ik_B$, and **vibration** at $\theta_{\text{vib}} = \hbar\omega/k_B$ far above it. Because
$\theta_{\text{rot}} \ll \theta_{\text{vib}}$, the heat capacity climbs a **staircase** —
$\tfrac32 \to \tfrac52 \to \tfrac72$ in units of $Nk_B$ — that nineteenth-century chemistry
measured and could not explain: equipartition, which cannot switch anything off, demands
$\tfrac72$ at every temperature, and the stubborn room-temperature $\tfrac52$ of real diatomics
was what Maxwell in 1875 called the gravest difficulty of the molecular theory. The resolution is
the freezing of [§7.5](oscillator-at-temperature.ipynb), now with two thresholds, and we assemble the full curve for H₂ and walk its
plateaus numerically.

The rotor itself deserves (and gets) its own treatment, because it is richer than the
oscillator in two instructive ways. Its level spacing *grows* with $J$, so its heat capacity does
not merely rise to the classical value but **overshoots** it — a hump peaking at $1.098\,k_B$
near $T = 0.81\,\theta_{\text{rot}}$, a calorimetric fingerprint as diagnostic as the Schottky
bump of [§7.4](thermal-density-matrix.ipynb). And its classical limit carries a measurable memory of discreteness: $z \to T/\theta +
\tfrac13 + \theta/15T + \dots$ (**Mulholland's** Euler–Maclaurin bridge), the rotor's version of
the Wigner correction of [§7.5](oscillator-at-temperature.ipynb).

The centerpiece is hydrogen. The two protons are identical spin-$\tfrac12$ fermions, so the total
wavefunction must be antisymmetric under their exchange ([§6.20](../06-quantum-mechanics/identical-particles.ipynb)) — and since exchange acts on the
rotational factor as the parity $(-1)^J$ and on the nuclear-spin factor as $\pm1$ (triplet or
singlet, [§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb)), the Pauli principle *deletes half of each ladder*: **para**-hydrogen is even-$J$
with the nuclear singlet, **ortho**-hydrogen is odd-$J$ with the triplet, weight 3. The
restricted sums deliver three results in sequence: the equilibrium ortho fraction ($\tfrac34$ at
high temperature ("normal hydrogen") and $0$ at low); the classical **symmetry number**
$\sigma = 2$, *derived* rather than decreed, the movement's second classical convention turned
theorem, after the $h$ of [§7.5](oscillator-at-temperature.ipynb); and **Dennison's 1927 resolution** of the great H₂ heat-capacity
puzzle — nuclear-spin conversion is so slow that cooled hydrogen keeps its room-temperature 3:1
composition, so experiment measures a *frozen mixture* utterly different from the equilibrium
curve, and taking the Pauli postulate seriously (rather than collecting better data) is what
closed a decade of confusion and confirmed proton spin from calorimetry. The engineering coda
prices all this in dollars: normal liquid hydrogen's ortho→para conversion heat *exceeds* its
vaporization enthalpy, so an uncatalyzed tank boils itself away — which is why every LH₂ plant,
and every rocket that flies on hydrogen, runs the liquefaction over a spin-conversion catalyst.

> **Conventions (this notebook).** Working units $k_B = 1$ with temperatures in Kelvin wherever a
> real molecule is on stage; molar restorations ($R = 8.314$ J mol⁻¹K⁻¹) appear in the Dennison
> and LH₂ sections. Rotor sums state their $J_{\max}$ and obey the convergence rule $J_{\max} \gg
> \sqrt{T/\theta}$ (the occupied levels reach $J \sim \sqrt{T/\theta}$ — like the trap of [§7.5](oscillator-at-temperature.ipynb), it bites
> at *high* temperature). Heat capacities come from the two-derivative chain $C =
> \partial_T\big(T^2\,\partial_T\ln z\big)$ by central differences with stated relative steps;
> the finite-difference dust this leaves at deep-freeze temperatures (occasional $10^{-10}$-level
> negatives) is noted where it appears, not hidden. Exercises 1–4 treat the rotor as
> *heteronuclear* (no exchange symmetry); the homonuclear physics enters deliberately with
> ortho/para in Exercise 5.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the rotor sum converging under its stated $J_{\max}$ rule (and failing without it);
> Mulholland's $\tfrac13 + \theta/15T$ against the measured $z - T/\theta$; the hump's location
> and height from `minimize_scalar` against the known $0.807\,\theta$ and $1.098\,k_B$; the
> staircase waypoints at 20/50/300/6000 K; the equilibrium ortho fractions at three temperatures;
> $z_{\text{even}}/z_{\text{full}} \to \tfrac12$ to four digits; the Dennison pair ($2.07\,k_B$
> equilibrium vs $0.004\,k_B$ frozen at 50 K); and the conversion-to-vaporization ratio $1.21$.
> A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** The diatomic gas as switches in series, and hydrogen's exchange physics. Vibration is
> *invoked* from [§7.5](oscillator-at-temperature.ipynb) (its `heat_capacity` restated in Setup, not re-derived); translation from
> [§5.6](../05-classical-stat-mech/ideal-gas-fundamental-relation.ipynb); the proper ensemble derivation of the statistics is [§7.7](bose-einstein-fermi-dirac.ipynb); coupled oscillators and Debye are
> [§7.16](phonons-debye.ipynb). See Pathria & Beale (Ch. 6); McQuarrie, *Statistical Mechanics* (Chs. 6, 8); Dennison
> (1927). Cross-reference [§6.14](../06-quantum-mechanics/angular-momentum-algebra.ipynb)/[§6.15](../06-quantum-mechanics/orbital-angular-momentum.ipynb) (the angular-momentum spectrum and its $2J+1$), [§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb) (the
> singlet and triplet), [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (exchange antisymmetry, applied here to nuclei), [§7.5](oscillator-at-temperature.ipynb) (the vibrational
> switch and the convention-to-theorem arc), [§5.6](../05-classical-stat-mech/ideal-gas-fundamental-relation.ipynb) (translation), and forward to [§7.7](bose-einstein-fermi-dirac.ipynb), [§7.8](classical-limit-thermal-wavelength.ipynb), [§7.16](phonons-debye.ipynb).

## Theory in brief

### The rigid rotor at temperature

A rigid diatomic rotor has the angular-momentum spectrum of [§6.14](../06-quantum-mechanics/angular-momentum-algebra.ipynb)/[§6.15](../06-quantum-mechanics/orbital-angular-momentum.ipynb): levels labelled by
$J = 0, 1, 2, \dots$ with energy and degeneracy

```{math}
:label: eq-rotor-Z
E_J = \frac{\hbar^2}{2I}J(J+1) \equiv k_B\theta_{\text{rot}}\,J(J+1),
\qquad
g_J = 2J+1,
\qquad
z_{\text{rot}} = \sum_{J} (2J+1)\,e^{-\theta_{\text{rot}}J(J+1)/T},
```

where the $2J+1$ *is* the $m$-multiplet of a given $J$ and $\theta_{\text{rot}} = \hbar^2/2Ik_B$
is the rotational quantum expressed as a temperature. The scales decide who gets to see any of
this: $\theta_{\text{rot}} = 87.5$ K for H₂, $15.2$ K for HCl, $2.9$ K for N₂, $2.1$ K for O₂.
The $1/I$ is doing the work — for every gas but hydrogen the rotational quantum lies far below
the boiling point, so rotation looks classical in any laboratory. Hydrogen's tiny moment of
inertia makes it the one molecule whose rotational quantum *shows*, which is precisely why it
became the battleground below.

### The classical limit and Mulholland's correction

At high temperature the sum blurs into an integral: substituting $u = J(J+1)\theta/T$ turns
$\sum(2J+1)e^{-\theta J(J+1)/T}$ into $(T/\theta)\int_0^\infty e^{-u}du$, so $z \to T/\theta$ —
two classical rotational degrees of freedom, each worth $\tfrac12 k_BT$. The *approach* is
measurable and has a name:

```{math}
:label: eq-mulholland
z_{\text{rot}} = \frac{T}{\theta} + \frac{1}{3} + \frac{\theta}{15\,T} + \mathcal{O}(\theta^2/T^2),
```

Mulholland's Euler–Maclaurin expansion. The discreteness never fully leaves $z$: it survives as
an additive $\tfrac13$ (and a $\theta/15T$ tail), exactly as $\hbar$ survived in the $(\hbar\omega)^2/12k_BT$
of [§7.5](oscillator-at-temperature.ipynb). We measure both terms.

### The rotational hump

The heat capacity follows from the two-derivative chain $C = \partial_T(T^2\,\partial_T\ln z)$,
and its shape is a fingerprint:

```{math}
:label: eq-rotational-hump
C_{\text{rot}}(T) \;\approx\; 12\,k_B\left(\frac{\theta}{T}\right)^2 e^{-2\theta/T}
\quad(T \ll \theta),
\qquad
C_{\text{rot}}\big(0.807\,\theta\big) = 1.098\,k_B
\quad(\text{the hump}),
\qquad
C_{\text{rot}} \to k_B \quad(T \gg \theta).
```

Frozen below $\theta$ (the first gap is $E_1 - E_0 = 2k_B\theta$, whence the $e^{-2\theta/T}$),
classical far above — but in between the rotor **overshoots** the plateau it will settle on. The
reason is the spacing law: rotor gaps $2\theta(J+1)$ *grow* with $J$, unlike the oscillator's
even rungs, so just past freezing the ladder is locally denser than the classical continuum
"expects" and the system briefly absorbs heat super-classically before the widening gaps rein it
in. Like the Schottky bump of [§7.4](thermal-density-matrix.ipynb), the hump is spectroscopy done with a calorimeter.

### The staircase

Wiring the switches in series gives the diatomic heat capacity,

```{math}
:label: eq-staircase
\frac{C_V}{Nk_B} = \underbrace{\frac{3}{2}}_{\text{translation (§5.6)}}
+ \underbrace{\frac{C_{\text{rot}}(T/\theta_{\text{rot}})}{k_B}}_{\text{this notebook}}
+ \underbrace{\frac{C_{\text{vib}}(T/\theta_{\text{vib}})}{k_B}}_{\text{oscillator (§7.5)}},
```

and for H₂ ($\theta_{\text{rot}} = 87.5$ K, $\theta_{\text{vib}} = 6300$ K) the curve climbs
$\tfrac32 \to \tfrac52 \to \tfrac72$ with room temperature sitting squarely on the $\tfrac52$
step. Equipartition demands $\tfrac72$ at *all* temperatures; the measured $\tfrac52$ is what
Maxwell named the gravest difficulty of the molecular theory, and it fell to nothing more exotic
than quantized level spacing. Honesty note: real H₂ dissociates near 3800 K, so the $\tfrac72$
plateau is asymptotic for the model and unreachable for the molecule — the curve is still exact
for the model, and N₂/O₂ tell the same story at their own $\theta$'s.

### Ortho and para hydrogen: Pauli dictates the spectrum

H₂'s two protons are identical spin-$\tfrac12$ fermions, so the total wavefunction must change
sign under their exchange ([§6.20](../06-quantum-mechanics/identical-particles.ipynb)). Exchange acts on the rotational factor as parity $(-1)^J$ and
on the nuclear-spin factor as $+1$ (triplet, symmetric) or $-1$ (singlet, antisymmetric) — the
two-spin algebra of [§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb), applied to nuclei. Antisymmetry of the product therefore *forces the pairing*:

```{math}
:label: eq-ortho-para
\text{para} = \{\text{even }J\}\otimes\text{singlet}\ (\text{weight }1),
\qquad
\text{ortho} = \{\text{odd }J\}\otimes\text{triplet}\ (\text{weight }3),
\qquad
x_{\text{ortho}}^{\text{eq}}(T) = \frac{3z_o}{z_p + 3z_o}.
```

Half of each species' rotational ladder is not suppressed but **absent** — the Pauli principle
legislating a thermodynamic spectrum. The equilibrium ortho fraction runs from $\tfrac34$ at high
temperature (the 3:1 "normal hydrogen" of the nuclear weights) to $0$ at low (everything into
$J = 0$, which is para).

### The symmetry number, derived

Classical statistical mechanics divides a homonuclear rotor's $z$ by $\sigma = 2$ "because the
molecule looks the same after half a turn." The quantum bookkeeping derives the rule:

```{math}
:label: eq-symmetry-number
\frac{z_{\text{even}}}{z_{\text{full}}} \xrightarrow{\;T\gg\theta\;} \frac12,
\qquad
z_p + 3z_o \;\longrightarrow\; 4\cdot\frac{z_{\text{full}}}{2}
= (\text{nuclear degeneracy})\times\frac{z_{\text{full}}}{\sigma},\quad \sigma = 2 .
```

Each parity class holds exactly half the classical states at high temperature, so the classical
fudge factor is a quantum theorem. This is the movement's ledger filling up: [§7.5](oscillator-at-temperature.ipynb) derived the $h$
in the measure, this notebook derives the $\sigma$, and [§7.8](classical-limit-thermal-wavelength.ipynb) will derive the $N!$.

### Dennison's resolution: equilibrium vs frozen

The 1920s crisis: measured H₂ heat capacities disagreed with the (correct!) equilibrium
calculation. The missing physics is *kinetic* — ortho↔para conversion requires a nuclear spin
flip, which ordinary collisions cannot provide, so cooled hydrogen keeps its room-temperature 3:1
composition on laboratory timescales. Experiment therefore measures the **frozen mixture** while
theory, innocently, computed **equilibrium**:

```{math}
:label: eq-dennison
C_{\text{frozen}} = \tfrac14 C_{\text{para}} + \tfrac34 C_{\text{ortho}},
\qquad
C_{\text{eq}} = \partial_T\Big(T^2\,\partial_T \ln\big(z_p + 3z_o\big)\Big),
```

and the two are not close: at 50 K the equilibrium curve carries a large conversion peak
($2.07\,k_B$ — the composition itself changes with $T$, a reaction-like heat) while the frozen
mixture is essentially dead ($0.004\,k_B$). Dennison's 1927 insight — posit the frozen ratio —
reconciled theory and experiment at a stroke, and stands as one of the earliest confirmations of
proton spin-$\tfrac12$ and Fermi statistics, delivered by calorimetry. Paramagnetic catalysts,
which do provide the spin flip, restore the equilibrium curve.

### The engineering coda: why rocket fuel is catalyzed

Every ortho molecule relaxing to para releases its $J{=}1$ rotational energy, $2k_B\theta_{\text
{rot}}$ per molecule:

```{math}
:label: eq-lh2
q_{\text{conv}} = 0.75 \times 2R\,\theta_{\text{rot}} \approx 1091\ \text{J mol}^{-1}
\;>\;
L_{\text{vap}}(\text{H}_2) \approx 899\ \text{J mol}^{-1},
```

so liquefied *normal* hydrogen carries more internal conversion heat than its own latent heat of
vaporization: left uncatalyzed, the tank slowly converts, self-heats, and boils away its entire
inventory. Every LH₂ plant runs the gas over an ortho–para catalyst during liquefaction. Quantum
statistics, priced in boil-off.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize_scalar

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: k_B = 1 (heat capacities per molecule, in units of k_B);
# temperatures in Kelvin whenever a named molecule is on stage; molar restorations via
# R = 8.314 J/(mol K) in the Dennison/LH2 sections. Rotor sums state J_max and obey
# J_max >> sqrt(T/θ) (occupied levels reach J ~ sqrt(T/θ): the trap bites at HIGH T).
# Finite-difference steps are RELATIVE (h = 1e-3 T), stated in C_from_lnz.

THETA_ROT_H2 = 87.5  # K — hydrogen's rotational quantum
THETA_VIB_H2 = 6300.0  # K — hydrogen's vibrational quantum
R_GAS = 8.314  # J/(mol K)


def z_rotor(T, theta, parity=None, J_max=None):
    """The rigid-rotor partition function Σ (2J+1) e^(−θ J(J+1)/T), optionally parity-restricted.

    Implements eq-rotor-Z as a degeneracy-weighted sum over a numpy.arange of J,
    with boolean parity masking selecting the even (para) or odd (ortho) ladder when
    requested — the restriction is Pauli's, not thermal (Exercise 5). J_max defaults
    to ceil(8·sqrt(T/θ)) + 20: the occupied levels reach J ~ sqrt(T/θ), so the tail
    beyond the default carries weight below e^(−60) at any T; Exercise 1 demonstrates
    the failure of an undersized J_max at high temperature.

    Parameters
    ----------
    T : float
        Temperature (same units as theta).
    theta : float
        The rotational quantum θ_rot = ħ^2/(2 I k_B).
    parity : str, optional
        None for the full ladder (default), 'even' for para, 'odd' for ortho.
    J_max : int, optional
        Truncation; None (default) applies the stated rule.

    Returns
    -------
    float
        The (restricted) rotational partition function.
    """
    if J_max is None:
        J_max = int(np.ceil(8.0 * np.sqrt(T / theta))) + 20
    J = np.arange(0, J_max + 1)
    if parity == "even":
        J = J[J % 2 == 0]
    elif parity == "odd":
        J = J[J % 2 == 1]
    elif parity is not None:
        raise ValueError("parity must be None, 'even', or 'odd'")
    J = J.astype(float)
    return float(np.sum((2.0 * J + 1.0) * np.exp(-theta * J * (J + 1.0) / T)))


def C_from_lnz(lnz, T, h_rel=1e-3):
    """The heat capacity C/k_B = ∂_T(T^2 ∂_T ln z) by the two-derivative chain.

    Expanding the outer derivative: C = 2T·(ln z)' + T^2·(ln z)'', with both
    derivatives taken by central differences at the RELATIVE step h = h_rel·T (stated
    convention: h_rel = 1e-3). The double differentiation amplifies rounding — the
    second difference carries an eps/h^2 noise floor — so deep-freeze values below
    ~1e-9 are finite-difference dust (occasionally negative by ~1e-10); Exercise 3
    works above that floor and says so.

    Parameters
    ----------
    lnz : callable
        ln z as a function of temperature.
    T : float
        Temperature at which to evaluate C.
    h_rel : float, optional
        Relative central-difference step (default 1e-3).

    Returns
    -------
    float
        The heat capacity per molecule, in units of k_B.
    """
    h = h_rel * T
    d1 = (lnz(T + h) - lnz(T - h)) / (2.0 * h)
    d2 = (lnz(T + h) - 2.0 * lnz(T) + lnz(T - h)) / h**2
    return 2.0 * T * d1 + T**2 * d2


def heat_capacity_vib(x):
    """The vibrational (oscillator) heat capacity C/k_B = x^2 e^x/(e^x − 1)^2, x = θ_vib/T.

    Restated from §7.5 (the ecp convention: every callable a solution uses lives in the
    notebook's own Setup), expm1-safe exactly as there: the numerator's e^x is
    expm1(x) + 1 and the denominator is expm1(x)^2, keeping the classical limit
    1 − x^2/12 free of cancellation and letting the frozen regime underflow benignly.

    Parameters
    ----------
    x : float or numpy.ndarray
        The gap-to-temperature ratio θ_vib/T.

    Returns
    -------
    float or numpy.ndarray
        C/k_B for the vibrational mode.
    """
    em = np.expm1(x)
    return x**2 * (em + 1.0) / em**2


def x_ortho_equilibrium(T, theta=THETA_ROT_H2):
    """The equilibrium ortho fraction x = 3 z_o/(z_p + 3 z_o) of hydrogen (eq-ortho-para).

    The nuclear-triplet weight 3 multiplies the odd-J (ortho) ladder and the singlet
    weight 1 the even-J (para) ladder — Pauli's pairing, implemented as parity-masked
    rotor sums. Limits: 3/4 at high T (the 3:1 'normal hydrogen' of the pure nuclear
    weights, all rotational ladders equally classical) and 0 at low T (everything
    condenses into J = 0, which is para).

    Parameters
    ----------
    T : float
        Temperature in the same units as theta (Kelvin for real hydrogen).
    theta : float, optional
        The rotational quantum (default: hydrogen's 87.5 K).

    Returns
    -------
    float
        The equilibrium ortho fraction.
    """
    z_p = z_rotor(T, theta, parity="even")
    z_o = z_rotor(T, theta, parity="odd")
    return 3.0 * z_o / (z_p + 3.0 * z_o)


def C_frozen_mixture(T, theta=THETA_ROT_H2):
    """The frozen-composition heat capacity (1/4) C_para + (3/4) C_ortho (eq-dennison).

    A mixture of FIXED composition: each species' rotational heat capacity is computed
    from its own restricted ln z, then combined with the frozen 1:3 weights — heat
    capacities of non-interconverting components add. This is what a 1920s calorimeter
    measured. It is NOT the equilibrium curve, which must differentiate the combined
    ln(z_p + 3 z_o) so that the temperature-dependent composition contributes its
    conversion heat (Exercise 7 computes both and stages the discrepancy).

    Parameters
    ----------
    T : float
        Temperature (Kelvin for real hydrogen).
    theta : float, optional
        The rotational quantum (default: hydrogen's).

    Returns
    -------
    float
        C/k_B per molecule of the frozen 25:75 mixture.
    """
    C_para = C_from_lnz(lambda t: np.log(z_rotor(t, theta, parity="even")), T)
    C_ortho = C_from_lnz(lambda t: np.log(z_rotor(t, theta, parity="odd")), T)
    return 0.25 * C_para + 0.75 * C_ortho


def staircase_C(T, theta_rot=THETA_ROT_H2, theta_vib=THETA_VIB_H2):
    """The diatomic heat capacity C_V/(N k_B): translation + rotation + vibration (eq-staircase).

    Three switches in series: the classical 3/2 of translation (Volume V §5.6 — its own
    quantum lies absurdly low), the full-ladder rotor by the two-derivative chain, and
    the oscillator curve of §7.5 for vibration. Heteronuclear treatment (no symmetry
    restriction): the ortho/para physics is deliberately kept out until Exercise 5.

    Parameters
    ----------
    T : float
        Temperature in Kelvin.
    theta_rot, theta_vib : float, optional
        The two quanta (defaults: hydrogen's 87.5 K and 6300 K).

    Returns
    -------
    float
        C_V/(N k_B) for the model diatomic.
    """
    C_rot = C_from_lnz(lambda t: np.log(z_rotor(t, theta_rot)), T)
    return 1.5 + C_rot + float(heat_capacity_vib(theta_vib / T))

## Exercise 1 — The rotor's partition function, and who gets to see it

The second quantum switch, and the physical scales that decide whether a molecule ever shows it.
Cite {eq}`eq-rotor-Z`.

1. Derive the level structure $E_J = k_B\theta_{\text{rot}}J(J+1)$ with degeneracy $2J+1$ from
   the angular-momentum spectrum of [§6.14](../06-quantum-mechanics/angular-momentum-algebra.ipynb)/[§6.15](../06-quantum-mechanics/orbital-angular-momentum.ipynb), and define $\theta_{\text{rot}} = \hbar^2/2Ik_B$.
2. Implement `z_rotor` as the degeneracy-weighted sum with parity masking of `numpy.arange`,
   stating $J_{\max}$, and demonstrate the $J_{\max} \gg \sqrt{T/\theta}$ convergence rule by
   showing an undersized truncation fail at high temperature.
3. Tabulate $\theta_{\text{rot}}$ for H₂ ($87.5$ K), HCl ($15.2$ K), N₂ ($2.9$ K), O₂ ($2.1$ K)
   and explain why only hydrogen's rotational freezing is observable above its boiling point.
4. Compute $S_{\text{rot}}(T)$ and confirm the third law holds ($S \to 0$ into the $J = 0$
   state) — Movement I's standing check, passed a third time. (Computation + one prose sentence.)

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    err_short > 1e-8 and err_rule < 1e-12,
    "the J_max rule: an undersized truncation fails at high T, the stated rule converges",
    f"J_max=25 err {err_short:.1e} vs rule err {err_rule:.1e}",
)
validate.check(
    S_cold < 1e-6 and abs(S_hot - (np.log(50.0) + 1.0)) < 5e-3,
    "S_rot → 0 at low T (third law, third system) and → ln(T/θ) + 1 classically",
    f"S(0.05θ) = {S_cold:.1e}, S(50θ) = {S_hot:.4f}",
)

## Exercise 2 — Mulholland's correction: the ramp remembers the rungs

The classical limit of the rotor, with its approach measured — the rotor's version of the
recovery of $h$ in [§7.5](oscillator-at-temperature.ipynb). Cite {eq}`eq-mulholland`.

1. Show the high-$T$ sum becomes the integral $z \to T/\theta$ (substitute
   $u = J(J+1)\theta/T$), identifying the two classical rotational degrees of freedom.
2. Compute $z - T/\theta$ numerically at $T/\theta = 5, 20, 100$ and confirm convergence to
   $\tfrac13$.
3. Compare against the Euler–Maclaurin (Mulholland) prediction $\tfrac13 + \theta/15T$ and
   confirm the next-order term to available precision.
4. Interpret (prose): the discreteness never fully vanishes from $z$ — it survives as a
   measurable additive constant, just as $\hbar$ survived in the $(\hbar\omega)^2/12k_BT$ of [§7.5](oscillator-at-temperature.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    mulholland_measured,
    mulholland_predicted,
    "Mulholland: the Euler–Maclaurin bridge z − T/θ = 1/3 + θ/(15T) from staircase to ramp",
    rtol=2e-3,
)
validate.close(
    mulholland_measured[-1] - 1.0 / (15.0 * 100.0),
    1.0 / 3.0,
    "the additive 1/3: discreteness survives the classical limit as a constant (θ/15T tail removed)",
    rtol=1e-4,
)

## Exercise 3 — The rotational hump

Between frozen and classical, the rotor overshoots — a fingerprint of widening level spacings.
Cite {eq}`eq-rotational-hump`.

1. Compute $C_{\text{rot}}(T)$ via the two-derivative chain `C_from_lnz` (central differences,
   relative step $10^{-3}$) across $T/\theta = 0.1$ to $10$.
2. Locate the maximum with `scipy.optimize.minimize_scalar`: $T^* = 0.807\,\theta$,
   $C_{\max} = 1.098\,k_B$ — above the classical value it later settles to.
3. Confirm the deep-freeze law $C \propto e^{-2\theta/T}$ (the first gap is $E_1 - E_0 =
   2k_B\theta$) by fitting $\ln(C\,T^2)$ against $1/T$ with `numpy.polyfit` (expect slope $-2$).
4. Explain the overshoot (prose): the spacing $2\theta(J+1)$ grows with $J$, so just past
   freezing the ladder is locally denser than the classical continuum "expects," and the system
   briefly absorbs heat super-classically — contrast the oscillator's even rungs, which produce
   no hump.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    [T_peak, C_peak],
    [0.807, 1.098],
    "the rotational hump: overshoot to 1.098 k_B at T = 0.807 θ_rot",
    rtol=5e-3,
)
validate.close(
    slope_freeze,
    -2.0,
    "the deep-freeze law C ∝ e^(−2θ/T): the first gap is E_1 − E_0 = 2k_Bθ",
    rtol=5e-3,
)

## Exercise 4 — The staircase, assembled

Three switches in series, and the nineteenth century's gravest difficulty resolved. Cite
{eq}`eq-staircase`.

1. Assemble `staircase_C(T)` $= \tfrac32$ (translation, [§5.6](../05-classical-stat-mech/ideal-gas-fundamental-relation.ipynb) invoked) $+\ C_{\text{rot}}/k_B +
   C_{\text{vib}}/k_B$ (the `numpy.expm1`-safe curve of [§7.5](oscillator-at-temperature.ipynb), restated in Setup), for H₂'s
   $\theta_{\text{rot}} = 87.5$ K and $\theta_{\text{vib}} = 6300$ K.
2. Verify the waypoints $1.54$ (20 K), $2.45$ (50 K), $2.502$ (300 K), $3.41$ (6000 K), and plot
   the staircase with both $\theta$'s marked.
3. State the history (prose): equipartition demands $\tfrac72$ always; the measured
   room-temperature $\tfrac52$ was Maxwell's declared gravest difficulty of the molecular
   theory — resolved by nothing but level spacing.
4. Add the honest caveats: real H₂ dissociates near 3800 K (the $\tfrac72$ plateau is asymptotic
   for the model, unreachable for the molecule), and air's N₂/O₂ sit at $\tfrac52$ for the same
   reasons with their own $\theta$'s.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    waypoints_C,
    [1.54, 2.45, 2.502, 3.41],
    "the diatomic staircase 3/2 → 5/2 → 7/2: waypoints at 20, 50, 300, 6000 K",
    rtol=1e-2,
)
validate.check(
    abs(staircase_C(300.0) - 2.5) < 5e-3,
    "room temperature sits ON the 5/2 plateau — Maxwell's difficulty, resolved by level spacing",
    f"C(300 K) = {staircase_C(300.0):.4f}",
)

## Exercise 5 — Pauli deletes half the ladder: ortho and para hydrogen

Exchange symmetry, acting on nuclei, decides which rotational states exist at all. Cite
{eq}`eq-ortho-para`.

1. Derive the selection: proton-exchange antisymmetry ([§6.20](../06-quantum-mechanics/identical-particles.ipynb)) $\times$ rotational parity
   $(-1)^J$ $\times$ nuclear singlet/triplet symmetry ([§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb)) forces para $=$ even $J$ $\otimes$
   singlet (weight 1) and ortho $=$ odd $J$ $\otimes$ triplet (weight 3).
2. Implement $z_{\text{para}}$ and $z_{\text{ortho}}$ as parity-restricted sums (the `parity`
   masking of `z_rotor`) and compute the equilibrium fraction $x_{\text{ortho}}(T) =
   3z_o/(z_p + 3z_o)$.
3. Verify the limits: $0.0014$ at 20 K, $0.4798$ at 77 K, $0.7492$ at 300 K, $0.7500$ at
   1000 K — the 3:1 "normal hydrogen" of high temperature and the all-para ground state of low.
4. Reflect (prose): these levels are not suppressed but *absent* — the Pauli principle
   legislating a thermodynamic spectrum, the abstract postulate of [§6.20](../06-quantum-mechanics/identical-particles.ipynb) now visible in a gas's heat
   capacity.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [x_values[0], x_values[2], x_values[3]],
    [0.0014, 0.7492, 0.7500],
    "Pauli's bookkeeping: the equilibrium ortho fraction at 20, 300, 1000 K",
    rtol=2e-2,
)
validate.close(
    x_values[1],
    0.4798,
    "the operationally loaded point: 48% ortho at liquid-nitrogen temperature",
    rtol=2e-2,
)

## Exercise 6 — The symmetry number, derived

The classical $\sigma = 2$ for homonuclear molecules is a quantum theorem — the next of Volume V's
conventions to be derived. Cite {eq}`eq-symmetry-number`.

1. Compute $z_{\text{even}}/z_{\text{full}}$ at $T/\theta = 3, 10, 50$ and confirm the approach
   to $\tfrac12$.
2. Show the consequence: $(z_p + 3z_o) \to 4\cdot(z_{\text{full}}/2)$ at high $T$, i.e.
   $(\text{nuclear degeneracy})\times(z_{\text{full}}/\sigma)$ with $\sigma = 2$.
3. State the classical rule as inherited by Volume V-style calculations ("divide by the number
   of indistinguishable orientations") and its now-derived status.
4. Place it in the ledger (prose): [§7.5](oscillator-at-temperature.ipynb) derived the $h$ in the measure, this exercise the
   $\sigma$, and [§7.8](classical-limit-thermal-wavelength.ipynb) will derive the $N!$ — the classical formalism's fudge factors, each a
   quantum limit.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    ratio_parity[50.0],
    0.5,
    "σ = 2 derived: each parity class carries exactly half the classical rotational states",
    rtol=1e-4,
)
validate.close(
    combined,
    classical_form,
    "(z_p + 3z_o) → (nuclear degeneracy) × z_full/σ: the classical recipe, now a theorem",
    rtol=1e-4,
)

## Exercise 7 — Dennison's resolution: the frozen mixture

Equilibrium theory versus a gas that cannot flip its nuclear spins — the 1920s crisis,
recomputed. Cite {eq}`eq-dennison`.

1. Compute the species curves $C_{\text{para}}(T)$ and $C_{\text{ortho}}(T)$ from the restricted
   sums, and the frozen mixture $C_{\text{frozen}} = \tfrac14 C_{\text{para}} + \tfrac34
   C_{\text{ortho}}$ (fixed composition: heat capacities add).
2. Compute the equilibrium curve $C_{\text{eq}}$ by differentiating the *combined*
   $\ln(z_p + 3z_o)$ — noting in the solution that mixing the species $C$'s here is precisely
   the error the distinction forbids.
3. Verify the drama at 50 K: $C_{\text{eq}} = 2.07\,k_B$ (the conversion peak — composition
   change acts like a reaction heat) vs $C_{\text{frozen}} = 0.004\,k_B$; plot all four curves.
4. Tell the resolution (prose): experiments on cooled H₂ traced the frozen curve; theory
   produced the equilibrium one; Dennison (1927) reconciled them by positing that nuclear-spin
   conversion is collisionally forbidden on laboratory timescales — calorimetry thereby
   confirming proton spin-$\tfrac12$ and Fermi statistics. Catalysts restore equilibrium.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    C_eq_50,
    2.07,
    "the equilibrium conversion peak: C_eq(50 K) = 2.07 k_B from the combined ln(z_p + 3z_o)",
    rtol=5e-2,
)
validate.check(
    C_frozen_50 < 5e-3 and C_eq_50 / C_frozen_50 > 400,
    "Dennison: the frozen mixture is dead (≈0.004 k_B) where equilibrium peaks — different curves",
    f"C_frozen(50 K) = {C_frozen_50:.4f}, ratio {C_eq_50 / C_frozen_50:.0f}×",
)
validate.check(
    float(np.max(np.abs(C_eq_curve[-20:] - C_frozen_curve[-20:]))) < 0.03,
    "above ~250 K the curves merge: composition stops moving and conversion heat vanishes",
)

## Exercise 8 — Why rocket fuel is catalyzed

The ortho–para conversion heat versus the latent heat of the liquid it sits in — quantum
statistics with an invoice. Cite {eq}`eq-lh2`.

1. Compute the $J = 1 \to 0$ conversion heat $2\theta_{\text{rot}}R$ per mole and the release
   for normal hydrogen's 75% ortho.
2. Compare against H₂'s vaporization enthalpy of $899$ J mol⁻¹: the ratio exceeds unity — the
   stored liquid carries more than enough internal conversion heat to boil itself entirely.
3. Estimate (order of magnitude, stated assumptions) the boil-off fraction of an uncatalyzed
   tank as conversion proceeds to completion, and note the industrial answer: catalyzed
   conversion *during* liquefaction.
4. Reflect (prose): the Pauli principle, two protons deep inside a molecule, sets the boil-off
   economics of every liquid-hydrogen rocket — the volume's most concrete "quantum statistics
   you can invoice."

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    ratio_heat_to_latent,
    1.21,
    "normal LH2 can boil itself away: conversion heat exceeds vaporization enthalpy",
    rtol=2e-2,
)
validate.close(
    q_J1,
    1455.0,
    "the J=1→0 conversion heat 2θ_rot R = 1455 J/mol — Pauli's stored energy, per mole",
    rtol=1e-2,
)

## Exercise 9 — Movement I, closed

Three notebooks, three exactly solvable systems, and the movement's ledger is full. The warm
qubit gave us the Fermi function, the Schottky bump, the third law, and temperatures beyond
infinity. The warm oscillator gave the Bose function, freezing out, the $h$ that classical
mechanics had borrowed, and a scandal about diamond closed to three digits. And the warm molecule
stacked the switches into a staircase that answered Maxwell's gravest difficulty, let the Pauli
principle delete half a spectrum, derived the symmetry number that classical mechanics had
decreed, and ended by explaining why rocket fuel needs a catalyst. Every classical failure on
Volume V's books (equipartition's blindness, the measure's mystery factors) has been explained by
discreteness, with one item (the $N!$) held over for [§7.8](classical-limit-thermal-wavelength.ipynb).

There is something delicious about the hydrogen story in particular. The most abstract principle
in the course, antisymmetry under exchange of identical fermions, reached into a bottle of gas,
deleted every other rotational level, split the substance into two species that ignore each other
for weeks, and confused a decade of excellent experimentalists. The confusion was resolved not by
better data but by taking the postulate seriously. Sometimes the bookkeeping *is* the physics.

Movement II now derives what Movement I kept meeting by accident: the Bose–Einstein and
Fermi–Dirac distributions, properly, from the grand canonical ensemble ([§7.7](bose-einstein-fermi-dirac.ipynb)) — the rendezvous
that three independent routes (a qubit, an oscillator, and the contours of [§7.2](complex-analysis-applications.ipynb)) have been promising
for five notebooks.

## Notebook summary

Movement I closes with the diatomic gas as a bank of quantum switches, and with identical-particle
physics rearranging a real substance's thermodynamics.

- **The rigid rotor** {eq}`eq-rotor-Z`: levels $k_B\theta J(J+1)$, degeneracy $2J+1$; the parity-
  maskable sum with its stated $J_{\max} \gg \sqrt{T/\theta}$ rule (failure demonstrated); only
  hydrogen's $\theta_{\text{rot}}$ clears its boiling point, so only hydrogen shows the quantum.
  Third law passed a third time.
- **Mulholland's correction** {eq}`eq-mulholland`: $z - T/\theta \to \tfrac13 + \theta/15T$,
  measured to the next order — discreteness surviving the classical limit as an additive
  constant, the rotor's Wigner term.
- **The rotational hump** {eq}`eq-rotational-hump`: freeze-out $\propto e^{-2\theta/T}$ (fitted
  slope $-2.00$), an overshoot to $1.098\,k_B$ at $0.807\,\theta$, then the classical plateau —
  the calorimetric fingerprint of a ladder whose rungs widen.
- **The staircase** {eq}`eq-staircase`: $\tfrac32 \to \tfrac52 \to \tfrac72$ for H₂'s two
  $\theta$'s, waypoints verified; room temperature sits on the $\tfrac52$ step that equipartition
  could not explain — Maxwell's gravest difficulty, resolved by level spacing (with the
  dissociation caveat stated honestly).
- **Ortho and para** {eq}`eq-ortho-para`: proton-exchange antisymmetry forces even-$J\,\otimes$
  singlet and odd-$J\,\otimes$ triplet; the equilibrium ortho fraction runs $\tfrac34 \to 0$
  (0.48 at 77 K) — Pauli legislating a thermodynamic spectrum.
- **The symmetry number** {eq}`eq-symmetry-number`: $z_{\text{even}}/z_{\text{full}} \to \tfrac12$
  to four digits, so the classical $\sigma = 2$ is a derived theorem — the tally now
  reads $h$ ([§7.5](oscillator-at-temperature.ipynb)), $\sigma$ (here), $N!$ ([§7.8](classical-limit-thermal-wavelength.ipynb), pending).
- **Dennison's resolution** {eq}`eq-dennison`: equilibrium ($2.07\,k_B$ at 50 K, conversion peak
  from the combined $\ln z$ — never from averaged species curves) versus the frozen mixture
  ($0.004\,k_B$): different curves, reconciled in 1927 by collisionally forbidden spin flips —
  proton spin and Fermi statistics, confirmed by calorimetry.
- **The invoice** {eq}`eq-lh2`: conversion heat $1091$ J/mol against latent heat $899$ J/mol
  (ratio $1.21$) — uncatalyzed liquid normal hydrogen boils itself away, and every LH₂ plant
  catalyzes during liquefaction.

Movement I is closed; the statistics get derived next.

## Outlook

- **The rendezvous ([§7.7](bose-einstein-fermi-dirac.ipynb)).** The grand canonical derivation of $n_B$ and $n_F$ — the qubit, the
  oscillator, and the contours of [§7.2](complex-analysis-applications.ipynb) converging on the same two functions.
- **The classical limit completed ([§7.8](classical-limit-thermal-wavelength.ipynb)).** The $N!$ derived at last, and the thermal de Broglie
  wavelength — when *is* a gas classical?
- **Solids ([§7.16](phonons-debye.ipynb)).** From independent switches to coupled modes: Debye, and the $T^3$ law the
  Einstein solid missed.
- **Horizons, named.** Polyatomic molecules and their symmetry numbers; hindered rotors;
  para-hydrogen-induced NMR hyperpolarization — today's spin bookkeeping as a laboratory tool.
- **Cross-reference** [§6.14](../06-quantum-mechanics/angular-momentum-algebra.ipynb)/[§6.15](../06-quantum-mechanics/orbital-angular-momentum.ipynb) (the rotor spectrum and its $2J+1$), [§6.19](../06-quantum-mechanics/addition-angular-momenta.ipynb)/[§6.20](../06-quantum-mechanics/identical-particles.ipynb) (singlet/triplet
  and exchange antisymmetry, applied here to nuclei), [§7.5](oscillator-at-temperature.ipynb) (the vibrational switch and the
  convention-to-theorem arc), [§5.6](../05-classical-stat-mech/ideal-gas-fundamental-relation.ipynb) (translation), [§7.4](thermal-density-matrix.ipynb) (the standing third-law check).

In [ ]:
from ecp.style import footer

footer()